## DistilBERT Classifier Evaluation

First, we need to install the `transformers` and `datasets` libraries, which are essential for working with pre-trained models and datasets from Hugging Face. We'll also install `evaluate` for easily computing metrics.

In [ ]:
!pip install transformers datasets evaluate



Next, we'll import the necessary libraries and load a suitable dataset for text classification. The [SST-2 (Stanford Sentiment Treebank v2)](https://huggingface.co/datasets/glue) dataset is a common choice for sentiment analysis, which is a binary classification task. We'll load the validation split for evaluation.

In [ ]:
from datasets import load_dataset, Dataset, ClassLabel
from transformers import AutoTokenizer, DistilBertForSequenceClassification
import torch
import evaluate
import numpy as np
import pandas as pd

# Load the user's combined_test.parquet dataset
df = pd.read_parquet("test.parquet")
dataset = Dataset.from_pandas(df)

# Get unique labels and create a label mapping
unique_labels = sorted(df['label'].unique().tolist())
label_to_id = {label: i for i, label in enumerate(unique_labels)}

# Map string labels to integer IDs and cast the 'label' column to ClassLabel type
dataset = dataset.map(lambda examples: {'label': label_to_id[examples['label']]}, batched=False)
dataset = dataset.cast_column('label', ClassLabel(names=unique_labels))

# Display some information about the dataset and its features
print(dataset)

We'll then load the `DistilBERT` tokenizer and preprocess the dataset. This involves tokenizing the text and converting labels to tensors. We'll also define a `data_collator` to handle dynamic padding during batching.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    # Using the correct text column name from your dataset: 'statement'
    return tokenizer(examples["statement"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Remove original text column and rename 'label' to 'labels' for consistency with model inputs
# Using the correct text column name: 'statement'
tokenized_datasets = tokenized_datasets.remove_columns(["statement"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# The DataCollatorWithPadding will handle setting the format to PyTorch tensors
# and padding, so we don't need to call .set_format('torch') here.
# tokenized_datasets.set_format("torch") # Removed this line

from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create DataLoader for the user's test dataset
# Renaming eval_dataloader to test_dataloader for clarity
test_dataloader = DataLoader(
    tokenized_datasets, batch_size=16, collate_fn=data_collator
)

# Check a batch
for batch in test_dataloader:
    print({k: v.shape for k, v in batch.items()})
    break

Now, let's load a pre-trained `DistilBertForSequenceClassification` model. For the baseline evaluation *before fine-tuning*, we'll use a model with a randomly initialized classification head. This demonstrates how the model performs without any task-specific training.

In [ ]:
# num_labels should be derived from the actual dataset used
num_labels = tokenized_datasets.features["labels"].num_classes
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

# Ensure the model is in evaluation mode
model.eval()

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

### Baseline Evaluation (Before Fine-tuning)

Now we will run the evaluation on the `test_dataloader` using the pre-trained `DistilBERT` model. Since the classification head is randomly initialized, we expect the performance to be close to random chance. We will calculate accuracy and F1-score.

In [ ]:
# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# Lists to store predictions and true labels
all_predictions = []
all_labels = []

print("Starting baseline evaluation...")

# Evaluate the model
for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

print("Baseline evaluation complete.")

In [ ]:
# Compute metrics
accuracy = accuracy_metric.compute(predictions=all_predictions, references=all_labels)
f1 = f1_metric.compute(predictions=all_predictions, references=all_labels, average='weighted') # Using weighted for multi-class

print(f"Baseline Accuracy: {accuracy['accuracy']:.4f}")
print(f"Baseline F1-Score (weighted): {f1['f1']:.4f}")

print("\nThis represents the performance of DistilBERT as a classifier before any fine-tuning on your specific dataset. As expected, with a randomly initialized classification head, the performance is likely close to random.")

### Fine-tuned Model Evaluation

Now, let's load the fine-tuned `distilBERT_mentalhealth_detection` model and evaluate its performance on the same test set. We expect significantly better results as this model has been trained for the specific task.

In [ ]:
# Upgrade torchao to a compatible version as required by peft
!pip install --upgrade "torchao>=0.16.0"

from peft import PeftModel
from transformers import DistilBertForSequenceClassification
import torch

# Load the fine-tuned model from Hugging Face
fine_tuned_model_path = "NathanSJSU01/distilBERT_mentalhealth_detection"

# Step 1: Load the original base model
base_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels
)

# Step 2: Apply the LoRA adapters on top
fine_tuned_model = PeftModel.from_pretrained(
    base_model, fine_tuned_model_path
)

# Ensure the model is in evaluation mode
fine_tuned_model.eval()

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fine_tuned_model.to(device)
print(f"Using device: {device}")

In [ ]:
# Load metrics
accuracy_metric_ft = evaluate.load("accuracy")
f1_metric_ft = evaluate.load("f1")

# Lists to store predictions and true labels for fine-tuned model
all_predictions_ft = []
all_labels_ft = []

print("Starting fine-tuned model evaluation...")

# Evaluate the fine-tuned model
for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = fine_tuned_model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    all_predictions_ft.extend(predictions.cpu().numpy())
    all_labels_ft.extend(batch["labels"].cpu().numpy())

print("Fine-tuned model evaluation complete.")

In [ ]:
# Compute metrics for the fine-tuned model
accuracy_ft = accuracy_metric_ft.compute(predictions=all_predictions_ft, references=all_labels_ft)
f1_ft = f1_metric_ft.compute(predictions=all_predictions_ft, references=all_labels_ft, average='weighted') # Using weighted for multi-class

print(f"Fine-tuned Model Accuracy: {accuracy_ft['accuracy']:.4f}")
print(f"Fine-tuned Model F1-Score (weighted): {f1_ft['f1']:.4f}")

print("\nThis shows the performance of the fine-tuned DistilBERT model after training on your specific dataset. The metrics should be significantly higher than the baseline.")

### Detailed Classification Report

Let's get a more granular view of the fine-tuned model's performance by examining the precision, recall, and F1-score for each class.

In [ ]:
from sklearn.metrics import classification_report

# Get the class names from the dataset (assuming tokenized_datasets still holds the info)
# Ensure unique_labels is accessible, or re-derive it if necessary
# If unique_labels is not directly available, we can get it from tokenized_datasets.features['labels'].names
class_names = tokenized_datasets.features['labels'].names

print("Classification Report for Fine-tuned Model:")
print(classification_report(all_labels_ft, all_predictions_ft, target_names=class_names))

### Confusion Matrix

A confusion matrix provides a visual summary of the model's performance by showing the number of correct and incorrect predictions for each class. This helps identify specific classes that are being misclassified.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(all_labels_ft, all_predictions_ft)

# Get class names for better visualization
class_names = tokenized_datasets.features['labels'].names

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, ax=ax)
ax.set_title("Confusion Matrix for Fine-tuned Model")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Percentage Improvement Calculation

Let's quantify the improvement of the fine-tuned model compared to the baseline model.

In [ ]:
# Extract baseline metrics
baseline_accuracy = accuracy['accuracy']
baseline_f1 = f1['f1']

# Extract fine-tuned metrics
fined_tuned_accuracy = accuracy_ft['accuracy']
fined_tuned_f1 = f1_ft['f1']

# Calculate percentage improvement for accuracy
accuracy_improvement = ((fined_tuned_accuracy - baseline_accuracy) / baseline_accuracy) * 100

# Calculate percentage improvement for F1-score
f1_improvement = ((fined_tuned_f1 - baseline_f1) / baseline_f1) * 100

print(f"Percentage Improvement in Accuracy: {accuracy_improvement:.2f}%")
print(f"Percentage Improvement in F1-Score: {f1_improvement:.2f}%")

